# Baseline 2 — LLM Hallucination Demo
**Goal:** Show that GPT-4o-mini **hallucinates** specific clinical details (frequency, channels, amplitude, duration)  
when generating EEG reports **without evidence grounding**.

**Pipeline:**
1. Load a seizure EDF → extract ground-truth features (freq, amplitude, spatial, temporal)
2. Call GPT-4o-mini with **no feature context** — only "seizure detected"
3. Parse atomic claims from the report
4. Cross-check each claim against ground-truth → label VERIFIED / HALLUCINATED
5. Compute hallucination rate

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')

# ── Make src/ importable from notebooks/ ───────────────────────────────────
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.signal import welch   # used directly in visualization cells
from openai import OpenAI

# ── NeuroScribe src imports ─────────────────────────────────────────────────
from src.data.loader import parse_summary_file, build_patient_manifest, load_recording
from src.features.extractor import extract_features
from src.llm.report_generator import generate_unverified_report
from src.llm.claim_verifier import extract_claims, verify_claim

sns.set_theme(style='darkgrid'); plt.rcParams['figure.dpi'] = 100

DATA_DIR    = '../data/chb-mit'
RESULTS_DIR = '../results/baseline2'
os.makedirs(RESULTS_DIR, exist_ok=True)

FS = 256

# ── OpenAI client ─────────────────────────────────────────────────────────
# Set your key: export OPENAI_API_KEY=sk-...
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', 'YOUR_KEY_HERE'))

print('Setup complete.')

---
## 1. Load a Seizure Recording

In [ ]:
# ── Load seizure recording via src.data.loader ─────────────────────────────
PATIENT     = 'chb01'
PATIENT_DIR = os.path.join(DATA_DIR, PATIENT)

recordings   = build_patient_manifest(PATIENT_DIR, patient_id=1)
seizure_recs = [r for r in recordings if r.has_seizure and os.path.exists(r.edf_path)]

TARGET_REC = seizure_recs[0]
TARGET_EDF = TARGET_REC.filename

recording = load_recording(TARGET_REC)
data      = recording.data
ch_names  = recording.channel_names
n_channels, n_samples = data.shape

# seizure_intervals as (onset, offset) tuples for downstream visualization
seizure_intervals = [(s.onset, s.offset) for s in TARGET_REC.seizures]

print(f'File     : {TARGET_EDF}')
print(f'Shape    : {data.shape}')
print(f'Seizures : {seizure_intervals}')

---
## 2. Ground-Truth Feature Extraction

In [ ]:
# ── Extract ground-truth features via src.features.extractor ───────────────
all_features = []
for i, (onset, offset) in enumerate(seizure_intervals):
    feat = extract_features(
        data, ch_names,
        onset_sec=onset, offset_sec=offset,
        patient=PATIENT, filename=TARGET_EDF, fs=FS,
    )
    all_features.append(feat)
    print(f'Seizure {i+1}: {onset:.0f}–{offset:.0f}s  |  '
          f'duration={feat["temporal"]["duration_sec"]}s  |  '
          f'dominant_freq={feat["frequency"]["dominant_hz"]}Hz  |  '
          f'top_ch={feat["spatial"]["most_active"]}  |  '
          f'rms={feat["amplitude"]["rms_uV"]}µV')

In [ ]:
# ── Visualise extracted features for seizure 1 ────────────────────────────
feat = all_features[0]
onset, offset = seizure_intervals[0]
start, end = int(onset*FS), int(offset*FS)
seg = data[:, start:end]

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig)

# 1. Top-3 channel waveforms
ax1 = fig.add_subplot(gs[0, :])
t   = np.arange(seg.shape[1]) / FS
top3 = [ch_names.index(c) for c in feat['spatial']['top3_channels']]
colors = ['tomato','steelblue','mediumseagreen']
for idx, (ci, col) in enumerate(zip(top3, colors)):
    ax1.plot(t, seg[ci] + idx*200, color=col, lw=0.8,
             label=f'{ch_names[ci]}  (RMS={np.sqrt((seg[ci]**2).mean()):.1f}µV)')
ax1.set_title(f'Top-3 Active Channels During Seizure  ({onset:.0f}–{offset:.0f}s)')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude (µV + offset)')
ax1.legend(fontsize=8)

# 2. PSD
ax2 = fig.add_subplot(gs[1, 0])
f_ax, psd = welch(seg[top3[0]], fs=FS, nperseg=FS*2)
ax2.semilogy(f_ax, psd, color='steelblue', lw=1.5)
ax2.axvline(feat['frequency']['dominant_hz'], color='red', linestyle='--',
            label=f'Dominant: {feat["frequency"]["dominant_hz"]} Hz')
for b,(lo,hi,c) in {'δ':(0.5,4,'purple'),'θ':(4,8,'blue'),'α':(8,13,'green'),'β':(13,30,'orange')}.items():
    ax2.axvspan(lo,hi,alpha=0.08,color=c,label=b)
ax2.set_xlim(0, 50); ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('Power')
ax2.set_title('PSD — Most Active Channel'); ax2.legend(fontsize=7)

# 3. Per-channel RMS bar
ax3  = fig.add_subplot(gs[1, 1])
rms  = np.sqrt((seg**2).mean(axis=1))
cols = ['tomato' if i in top3 else 'steelblue' for i in range(n_channels)]
ax3.bar(range(n_channels), rms, color=cols)
ax3.set_xlabel('Channel index'); ax3.set_ylabel('RMS (µV)')
ax3.set_title('Per-Channel RMS Amplitude  (red = top 3)')

plt.suptitle(f'Ground-Truth Features — {TARGET_EDF}  Seizure 1', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'features_seizure1.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nGround-truth feature dict:')
print(json.dumps(feat, indent=2))

---
## 3. GPT-4o-mini — Unverified Report (Baseline 2)

In [ ]:
# ── Generate unverified reports via src.llm.report_generator ───────────────
# GPT-4o-mini is given ONLY patient ID + "seizure detected" — no features.
# It must invent specific values → hallucinated claims.
reports = []
for i, feat in enumerate(all_features):
    print(f'Generating report for seizure {i+1}...')
    report_text = generate_unverified_report(client, feat)
    reports.append(report_text)
    print(f'  Done ({len(report_text)} chars)\n')

print('\n' + '='*60)
print('REPORT — SEIZURE 1 (UNVERIFIED / NO GROUNDING)')
print('='*60)
print(reports[0])

---
## 4. Claim Extraction & Hallucination Analysis

In [ ]:
# ── Claim extractor and verifier loaded from src.llm.claim_verifier ────────
# extract_claims(report_text) → list of {category, claim_text, value_lo, value_hi, unit}
# verify_claim(claim, feat)   → (verdict, gt_value, explanation)
#   verdict: 'VERIFIED' | 'HALLUCINATED' | 'UNVERIFIABLE'
print('Claim extractor and verifier imported from src.llm.claim_verifier.')

In [ ]:
# ── Run analysis for all seizure reports ──────────────────────────────────
all_results = []

for i, (report_text, feat) in enumerate(zip(reports, all_features)):
    claims = extract_claims(report_text)

    for claim in claims:
        verdict, gt_val, explanation = verify_claim(claim, feat)
        all_results.append({
            'Seizure':    i + 1,
            'Category':   claim['category'],
            'LLM Claim':  claim['claim_text'],
            'GT Value':   gt_val,
            'Verdict':    verdict,
            'Explanation': explanation,
        })

df_claims = pd.DataFrame(all_results)
print(f'Total claims extracted: {len(df_claims)}')
print(df_claims['Verdict'].value_counts().to_string())
print()
df_claims

In [ ]:
# ── Hallucination rate ────────────────────────────────────────────────────
verifiable = df_claims[df_claims['Verdict'] != 'UNVERIFIABLE']
n_total      = len(verifiable)
n_hallucinated = (verifiable['Verdict'] == 'HALLUCINATED').sum()
n_verified     = (verifiable['Verdict'] == 'VERIFIED').sum()
halluc_rate    = n_hallucinated / max(n_total, 1)

print('\n' + '='*50)
print('HALLUCINATION ANALYSIS — Baseline 2')
print('='*50)
print(f'  Total verifiable claims : {n_total}')
print(f'  VERIFIED                : {n_verified}')
print(f'  HALLUCINATED            : {n_hallucinated}')
print(f'  Hallucination Rate      : {halluc_rate:.1%}')
print()
print('  By category:')
print(verifiable.groupby(['Category','Verdict']).size().to_string())

# Save
df_claims.to_csv(os.path.join(RESULTS_DIR, 'hallucination_analysis.csv'), index=False)
print(f'\nSaved → {RESULTS_DIR}/hallucination_analysis.csv')

In [ ]:
# ── Visualise hallucination breakdown ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall pie
axes[0].pie([n_verified, n_hallucinated],
            labels=['VERIFIED', 'HALLUCINATED'],
            colors=['mediumseagreen','tomato'],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title(f'Overall Hallucination Rate\n(n={n_total} verifiable claims)')

# Per-category stacked bar
cat_counts = verifiable.groupby(['Category','Verdict']).size().unstack(fill_value=0)
cat_counts.plot(kind='bar', stacked=True,
                color={'VERIFIED':'mediumseagreen','HALLUCINATED':'tomato'},
                ax=axes[1], edgecolor='white')
axes[1].set_xlabel('Claim Category'); axes[1].set_ylabel('Count')
axes[1].set_title('Hallucinations per Claim Category')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(loc='upper right')

plt.suptitle('Baseline 2 — GPT-4o-mini Without Evidence Grounding', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'hallucination_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Side-by-side: LLM claim vs ground truth ───────────────────────────────
print('CLAIM-BY-CLAIM COMPARISON — SEIZURE 1')
print('='*90)
s1 = df_claims[df_claims['Seizure'] == 1][['Category','LLM Claim','GT Value','Verdict']]

for _, row in s1.iterrows():
    colour = '✓' if row['Verdict'] == 'VERIFIED' else '✗'
    print(f'  {colour} [{row["Category"]:10s}]  '
          f'LLM: "{row["LLM Claim"]:20s}"  |  '
          f'GT: {row["GT Value"]:20s}  |  {row["Verdict"]}')

---
## 5. Summary

In [ ]:
print('='*55)
print('BASELINE 2 — HALLUCINATION SUMMARY')
print('='*55)
print(f'  Model   : GPT-4o-mini (no evidence grounding)')
print(f'  Input   : only patient ID + "seizure detected"')
print(f'  Reports : {len(reports)} generated')
print()
print(f'  Hallucination rate : {halluc_rate:.1%}  ({n_hallucinated}/{n_total} claims)')
print()
print(f'  This confirms the core motivation for NeuroScribe:')
print(f'  LLMs generate clinically plausible but factually')
print(f'  incorrect EEG measurements when not grounded')
print(f'  in extracted signal evidence.')
print()
print(f'  → Next: Evidence Verification Agent (final system)')
print(f'    achieves 0% hallucination by grounding every')
print(f'    claim against the extracted feature JSON.')
print('='*55)